# AgentState & Trajectory
Agent 的状态散落在多个变量里：
```
response
tool_outputs
response.id
item.call_id
item.arguments
```
我们需要写一个 State 来维护 Agent 的状态—————即运行过程中需要记录的 Agent 的所有信息。包括：最初任务要求、迭代次数、上一轮调用的 Tool，response_id……

这一步很重要，后面将 经验转为 skill，就需要记录 Trajectory，Agent 依据 Trajectory 自我学习该任务是如何处理的，自我总结为自己的 skill

## 该部分主要内容
把原来散落在 Agent Loop 里的运行信息，统一收进 AgentState。

In [3]:
import json

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field


# ============================================================
# 1. Load environment variables
# ============================================================

load_dotenv()


# ============================================================
# 2. Create LLM client
# ============================================================

client = OpenAI()


# ============================================================
# 3. Agent State
# ============================================================

class AgentState(BaseModel):
    """
    保存 Agent 在整个任务执行过程中的状态。
    """

    # 用户最初提出的任务
    task: str

    # 当前 Agent 执行到了第几轮
    iteration: int = 0

    # Agent 是否已经完成任务
    finished: bool = False

    # LLM 请求过哪些工具
    tool_calls: list = Field(default_factory=list)

    # 工具实际返回了什么结果
    tool_results: list = Field(default_factory=list)


# ============================================================
# 4. Tools
# ============================================================

def calculator(
    a: float,
    b: float,
    operation: str,
) -> float:
    """
    执行基本数学计算。
    """

    if operation == "add":
        return a + b

    if operation == "subtract":
        return a - b

    if operation == "multiply":
        return a * b

    if operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero.")

        return a / b

    raise ValueError(
        f"Unknown operation: {operation}"
    )


def get_user_name() -> str:
    """
    获取当前用户的名字。
    """

    return "Sylvie"


# ============================================================
# 5. Tool Registry
# ============================================================

tool_registry = {
    "calculator": calculator,
    "get_user_name": get_user_name,
}


# ============================================================
# 6. Tool Schemas
# ============================================================

tools = [
    {
        "type": "function",
        "name": "calculator",
        "description": "执行基本数学计算。",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "第一个数字。",
                },
                "b": {
                    "type": "number",
                    "description": "第二个数字。",
                },
                "operation": {
                    "type": "string",
                    "enum": [
                        "add",
                        "subtract",
                        "multiply",
                        "divide",
                    ],
                    "description": "计算操作。",
                },
            },
            "required": [
                "a",
                "b",
                "operation",
            ],
        },
    },
    {
        "type": "function",
        "name": "get_user_name",
        "description": "获取当前用户的名字。",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
        },
    },
]


# ============================================================
# 7. Main Agent
# ============================================================

def run_agent(task: str) -> AgentState:

    # --------------------------------------------------------
    # Create Agent State
    # --------------------------------------------------------

    state = AgentState(
        # 初始化任务
        task=task
    )

    # --------------------------------------------------------
    # Initial input
    # --------------------------------------------------------

    input_items = [
        {
            "role": "user",
            "content": task,
        }
    ]

    # --------------------------------------------------------
    # First LLM call
    # --------------------------------------------------------

    response = client.responses.create(
        model="gpt-5",
        input=input_items,
        tools=tools,
    )

    # --------------------------------------------------------
    # Agent Loop
    # --------------------------------------------------------

    while not state.finished:

        # ----------------------------------------------------
        # Update iteration
        # ----------------------------------------------------

        state.iteration += 1

        print(
            f"\n--- Agent Iteration {state.iteration} ---"
        )

        # ----------------------------------------------------
        # Safety limit
        # ----------------------------------------------------

        if state.iteration > 10:
            raise RuntimeError(
                "Agent exceeded maximum iterations."
            )

        # ----------------------------------------------------
        # Store tool outputs that will be sent back to LLM
        # ----------------------------------------------------

        tool_outputs = []

        # ----------------------------------------------------
        # Process current LLM response
        # ----------------------------------------------------

        for item in response.output:

            # =================================================
            # LLM requested a tool
            # =================================================

            if item.type == "function_call":

                print(
                    f"Tool Call: {item.name}"
                )

                print(
                    f"Arguments: {item.arguments}"
                )

                # ------------------------------------------------
                # Record Tool Call in AgentState
                # ------------------------------------------------

                state.tool_calls.append(
                    {
                        "name": item.name,
                        "arguments": item.arguments,
                        "call_id": item.call_id,
                    }
                )

                # ------------------------------------------------
                # Find actual Python function
                # ------------------------------------------------

                tool_name = item.name

                if tool_name not in tool_registry:
                    raise ValueError(
                        f"Unknown tool: {tool_name}"
                    )

                tool = tool_registry[tool_name]

                # ------------------------------------------------
                # Parse arguments
                # ------------------------------------------------

                arguments = json.loads(
                    item.arguments
                )

                # ------------------------------------------------
                # Execute tool
                # ------------------------------------------------

                result = tool(
                    **arguments
                )

                print(
                    f"Tool Result: {result}"
                )

                # ------------------------------------------------
                # Record Tool Result in AgentState
                # ------------------------------------------------

                state.tool_results.append(
                    {
                        "call_id": item.call_id,
                        "tool": tool_name,
                        "result": result,
                    }
                )

                # ------------------------------------------------
                # Build function_call_output
                # ------------------------------------------------

                tool_outputs.append(
                    {
                        "type": "function_call_output",
                        "call_id": item.call_id,
                        "output": str(result),
                    }
                )

        # ========================================================
        # No Tool Call
        # ========================================================

        if not tool_outputs:

            state.finished = True

            print("\nFinal Answer:")
            print(response.output_text)

            break

        # ========================================================
        # Send Tool Results back to LLM
        # ========================================================

        response = client.responses.create(
            model="gpt-5",
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
        )

    # ============================================================
    # Return final Agent State
    # ============================================================

    return state


# ============================================================
# 8. Run Agent
# ============================================================

if __name__ == "__main__":

    task = (
        "请告诉我我的名字，然后计算 "
        "12345 + 6789，再把结果乘以 2。"
    )

    final_state = run_agent(task)

    # ========================================================
    # Print Agent State
    # ========================================================

    print("\n" + "=" * 60)
    print("FINAL AGENT STATE")
    print("=" * 60)

    print(
        f"\nTask:\n{final_state.task}"
    )

    print(
        f"\nIterations:\n{final_state.iteration}"
    )

    print(
        f"\nFinished:\n{final_state.finished}"
    )

    print(
        "\nTool Calls:"
    )

    for call in final_state.tool_calls:
        print(call)

    print(
        "\nTool Results:"
    )

    for result in final_state.tool_results:
        print(result)



--- Agent Iteration 1 ---
Tool Call: get_user_name
Arguments: {}
Tool Result: Sylvie
Tool Call: calculator
Arguments: {"a":12345,"b":6789,"operation":"add"}
Tool Result: 19134

--- Agent Iteration 2 ---
Tool Call: calculator
Arguments: {"a":19134,"b":2,"operation":"multiply"}
Tool Result: 38268

--- Agent Iteration 3 ---

Final Answer:
你的名字是 Sylvie。计算结果：12345 + 6789 = 19134，乘以 2 得到 38268。

FINAL AGENT STATE

Task:
请告诉我我的名字，然后计算 12345 + 6789，再把结果乘以 2。

Iterations:
3

Finished:
True

Tool Calls:
{'name': 'get_user_name', 'arguments': '{}', 'call_id': 'call_2OQcbVD1T0Jy3bewjFnbB4AV'}
{'name': 'calculator', 'arguments': '{"a":12345,"b":6789,"operation":"add"}', 'call_id': 'call_XmVYQHh1aZRRUshlHk2gy6IY'}
{'name': 'calculator', 'arguments': '{"a":19134,"b":2,"operation":"multiply"}', 'call_id': 'call_IOyaufSNaD83b9FafcLqd6AT'}

Tool Results:
{'call_id': 'call_2OQcbVD1T0Jy3bewjFnbB4AV', 'tool': 'get_user_name', 'result': 'Sylvie'}
{'call_id': 'call_XmVYQHh1aZRRUshlHk2gy6IY', 'tool': 'calcul